# Global Classifier Comparison — ASL Recognition Dissertation
**Author:** Owasu Brown | National University | 2025–2026

Compares four classifiers across four experimental conditions.
All tables follow APA 7 formatting conventions.
WEASEL+MUSE is excluded — reserved for the extension paper.

## Classifiers
- Random Forest (RF)
- Logistic Regression (LR)
- Canonical Interval Forest (CIF)
- InceptionTime (IT)

## Experimental conditions
1. **243-class** — reduced vocabulary, augmented, multiple signers
2. **1,683-class** — full vocabulary, leave-one-out split, multiple signers
3. **Signer-controlled** — signers 11 + 13 only, 201 classes, augmented
4. **Single-signer** — signer 11 only, 121 classes, 7× augmentation

## Run order
Cell 1 → 2 → 3 → 4 → 5 (tables) → 6 (bar charts) →
7 (top-N curves) → 8 (F1 distribution) → 9 (cross-experiment) → 10 (export)


In [1]:
import subprocess, sys
subprocess.check_call([
    sys.executable,'-m','pip','install',
    'pandas','numpy','scikit-learn','matplotlib',
    'joblib','scipy','-q'
])
print('✅ Packages ready')


✅ Packages ready


In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
import os, sys, json, warnings
import numpy as np
import pandas as pd
import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path
from sklearn.metrics import (
    accuracy_score, f1_score,
    precision_recall_fscore_support,
    top_k_accuracy_score
)
warnings.filterwarnings('ignore')

PROJECT_DIR    = '/content/drive/MyDrive/Consultant/Colab_Notebooks/Obrown_Dissertation_NU_25/OBrown_DIS9300_v2'
RESULTS_DIR    = os.path.join(PROJECT_DIR, 'results')
COMPARISON_DIR = os.path.join(RESULTS_DIR, 'global_comparison_v2')
os.makedirs(COMPARISON_DIR, exist_ok=True)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
print(f'Results dir  : {RESULTS_DIR}')
print(f'Output dir   : {COMPARISON_DIR}')


Results dir  : /content/drive/MyDrive/Consultant/Colab_Notebooks/Obrown_Dissertation_NU_25/OBrown_DIS9300_v2/results
Output dir   : /content/drive/MyDrive/Consultant/Colab_Notebooks/Obrown_Dissertation_NU_25/OBrown_DIS9300_v2/results/global_comparison_v2


In [4]:
# ── Experiment registry ──────────────────────────────────────────────────
# Each entry: (experiment_label, results_folder, model_class_name, pkl_name)
# WEASEL+MUSE excluded — reserved for extension paper.

EXPERIMENTS = [
    # ── 243-class ─────────────────────────────────────────────────────────
    ('243-class',        'rf_243',   'RandomForestClassifier',       'random_forest_model.pkl',       'Random Forest'),
    ('243-class',        'lr_243',   'LogisticRegressionClassifier', 'logistic_regression_model.pkl', 'Logistic Regression'),
    ('243-class',        'cif_243',  'CanonicalIntervalForest',      'cif_model.pkl',                 'CIF'),
    ('243-class',        'it_243',   'InceptionTimeClassifier',      'inceptiontime_model.pkl',       'InceptionTime'),
    # ── 1,683-class ───────────────────────────────────────────────────────
    ('1,683-class',      'rf_full',  'RandomForestClassifier',       'random_forest_model.pkl',       'Random Forest'),
    ('1,683-class',      'lr_full',  'LogisticRegressionClassifier', 'logistic_regression_model.pkl', 'Logistic Regression'),
    ('1,683-class',      'cif_full', 'CanonicalIntervalForest',      'cif_model.pkl',                 'CIF'),
    ('1,683-class',      'it_full',  'InceptionTimeClassifier',      'inceptiontime_model.pkl',       'InceptionTime'),
    # ── Signer-controlled (signers 11+13) ─────────────────────────────────
    ('Signer-controlled','signer_rf',  'RandomForestClassifier',       'random_forest_model.pkl',       'Random Forest'),
    ('Signer-controlled','signer_lr',  'LogisticRegressionClassifier', 'logistic_regression_model.pkl', 'Logistic Regression'),
    ('Signer-controlled','signer_it',  'InceptionTimeClassifier',      'inceptiontime_model.pkl',       'InceptionTime'),
    # ── Single-signer (signer 11) ─────────────────────────────────────────
    ('Single-signer',    's11_rf',   'RandomForestClassifier',       'random_forest_model.pkl',       'Random Forest'),
    ('Single-signer',    's11_lr',   'LogisticRegressionClassifier', 'logistic_regression_model.pkl', 'Logistic Regression'),
    ('Single-signer',    's11_cif',  'CanonicalIntervalForest',      'cif_model.pkl',                 'CIF'),
    ('Single-signer',    's11_it',   'InceptionTimeClassifier',      'inceptiontime_model.pkl',       'InceptionTime'),
]

# APA 7 colour palette — accessible, print-friendly
MODEL_COLORS = {
    'Random Forest'      : '#2E75B6',
    'Logistic Regression': '#ED7D31',
    'CIF'                : '#70AD47',
    'InceptionTime'      : '#7030A0',
}
MODEL_MARKERS = {
    'Random Forest'      : 'o',
    'Logistic Regression': 's',
    'CIF'                : '^',
    'InceptionTime'      : 'D',
}
EXP_ORDER = ['243-class','1,683-class','Signer-controlled','Single-signer']
MODEL_ORDER = ['InceptionTime','Random Forest','Logistic Regression','CIF']

# APA 7 matplotlib style
plt.rcParams.update({
    'font.family'      : 'serif',
    'font.size'        : 10,
    'axes.titlesize'   : 11,
    'axes.labelsize'   : 10,
    'legend.fontsize'  : 9,
    'axes.spines.top'  : False,
    'axes.spines.right': False,
    'axes.grid'        : False,
    'figure.dpi'       : 150,
})
print('✅ Configuration loaded')
print(f'   {len(EXPERIMENTS)} experiment-model combinations registered')


✅ Configuration loaded
   15 experiment-model combinations registered


In [5]:
# ── Load all saved artifacts ─────────────────────────────────────────────
records = []   # list of dicts — one per experiment × model combination
proba_store = {}  # (exp, folder) → y_proba

def find_file(base, pattern):
    for root, _, files in os.walk(base):
        for f in files:
            if f.endswith(pattern):
                return os.path.join(root, f)
    return None

for exp_label, folder, cls_name, pkl_name, model_label in EXPERIMENTS:
    base = os.path.join(RESULTS_DIR, folder)
    if not os.path.exists(base):
        print(f'  ⬜  {exp_label:20s} {model_label:22s} — folder not found')
        continue

    # Predictions CSV
    pred_path = find_file(base, '_predictions.csv')
    if pred_path is None:
        print(f'  ❌  {exp_label:20s} {model_label:22s} — predictions CSV missing')
        continue
    df_pred = pd.read_csv(pred_path)
    y_true  = df_pred['y_true'].values
    y_pred  = df_pred['y_pred'].values

    # Probabilities
    proba_path = find_file(base, '_proba.npy')
    y_proba = np.load(proba_path, allow_pickle=True) if proba_path else None
    proba_store[(exp_label, folder)] = y_proba

    # Per-class metrics
    pc_path = find_file(base, '_per_class_metrics.csv')
    pc_df   = pd.read_csv(pc_path) if pc_path else None

    # Global metrics JSON
    jpath = find_file(base, '_global_metrics.json')
    gm    = json.load(open(jpath)) if jpath else {}

    n_classes = len(np.unique(y_true))

    # Compute metrics
    acc    = accuracy_score(y_true, y_pred)
    f1_mac = f1_score(y_true, y_pred, average='macro',    zero_division=0)
    f1_w   = f1_score(y_true, y_pred, average='weighted', zero_division=0)

    top5 = float('nan')
    if y_proba is not None and n_classes >= 5:
        try:
            top5 = top_k_accuracy_score(
                y_true, y_proba, k=5,
                labels=list(range(y_proba.shape[1]))
            )
        except Exception:
            pass

    # Model file size
    pkl_path = os.path.join(base, pkl_name)
    size_mb  = (os.path.getsize(pkl_path)/(1024**2)
                if os.path.exists(pkl_path) else float('nan'))

    records.append({
        'Experiment'  : exp_label,
        'Model'       : model_label,
        'folder'      : folder,
        'Accuracy'    : round(acc,    4),
        'F1 Macro'    : round(f1_mac, 4),
        'F1 Weighted' : round(f1_w,   4),
        'Top-5 Acc'   : round(top5,   4) if not np.isnan(top5) else None,
        'N Classes'   : n_classes,
        'N Test'      : len(y_true),
        'Model MB'    : round(size_mb,2) if not np.isnan(size_mb) else None,
        '_y_true'     : y_true,
        '_y_pred'     : y_pred,
        '_pc_df'      : pc_df,
    })
    print(f'  ✅  {exp_label:20s} {model_label:22s} '
          f'acc={acc*100:5.2f}%  n={len(y_true)}')

df_all = pd.DataFrame(records)
print(f'\n{len(df_all)} results loaded successfully.')


  ✅  243-class            Random Forest          acc=12.06%  n=257
  ✅  243-class            Logistic Regression    acc= 9.73%  n=257
  ✅  243-class            CIF                    acc= 6.61%  n=257
  ✅  243-class            InceptionTime          acc=20.23%  n=257
  ✅  1,683-class          Random Forest          acc= 1.90%  n=1683
  ✅  1,683-class          Logistic Regression    acc= 0.30%  n=1683
  ✅  1,683-class          CIF                    acc= 1.01%  n=1683
  ✅  1,683-class          InceptionTime          acc= 6.71%  n=1683
  ✅  Signer-controlled    Random Forest          acc= 9.46%  n=222
  ✅  Signer-controlled    Logistic Regression    acc= 9.91%  n=222
  ✅  Signer-controlled    InceptionTime          acc=15.32%  n=222
  ✅  Single-signer        Random Forest          acc=16.22%  n=148
  ✅  Single-signer        Logistic Regression    acc=16.22%  n=148
  ✅  Single-signer        CIF                    acc=11.49%  n=148
  ✅  Single-signer        InceptionTime          acc=22.30

In [6]:

import matplotlib.gridspec as gridspec

def apa_table_image(df_exp, exp_label, out_dir):
    """Render one APA 7 style table as a PNG image."""
    cols  = ['Model','Accuracy','F1 Macro','F1 Weighted','Top-5 Acc',
              'N Classes','N Test']
    df_t  = df_exp[cols].copy()

    for c in ['Accuracy','F1 Macro','F1 Weighted','Top-5 Acc']:
        df_t[c] = df_t[c].apply(
            lambda x: f'{x*100:.2f}' if pd.notna(x) else '—'
        )
    df_t['N Classes'] = df_t['N Classes'].astype(str)
    df_t['N Test']    = df_t['N Test'].astype(str)

    col_labels = ['Model','Accuracy\n(%)',
                  'F1 Macro\n(%)','F1 Weighted\n(%)',
                  'Top-5\nAccuracy (%)','Classes','Test\nVideos']

    n_rows = len(df_t)
    # Figure height: title block (1.4 in) + header (0.4) + rows (0.35 each) + note (0.6)
    fig_h  = 1.4 + 0.4 + n_rows * 0.35 + 0.6
    fig, ax = plt.subplots(figsize=(13, fig_h))
    ax.axis('off')

    # Reserve space: top 1.4in for title, bottom 0.6in for note
    note_frac  = 0.6 / fig_h   # fraction for note at bottom
    title_frac = 1.4 / fig_h   # fraction for title block at top
    table_frac = 1.0 - title_frac - note_frac
    table_y0   = note_frac

    tbl = ax.table(
        cellText   = df_t.values,
        colLabels  = col_labels,
        cellLoc    = 'center',
        loc        = 'center',
        bbox       = [0, table_y0, 1, table_frac]
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(9)

    # Remove all cell borders
    for (row, col), cell in tbl.get_celld().items():
        cell.set_linewidth(0)
        cell.set_edgecolor('none')
        cell.set_facecolor('white')
        if row == 0:
            cell.set_text_props(fontweight='bold')

    # APA 7 horizontal rules
    y_top    = table_y0 + table_frac          # top of table
    y_header = table_y0 + table_frac * (n_rows / (n_rows + 1))  # below header
    y_bottom = table_y0                        # bottom of table
    for y in [y_top, y_header, y_bottom]:
        ax.axhline(y=y, xmin=0.01, xmax=0.99,
                   color='black', linewidth=0.8)

    # Title block — Table N on its own line, then italic title on next line
    # Positioned clearly above the top rule with a gap
    t_num = EXP_ORDER.index(exp_label) + 1 if exp_label in EXP_ORDER else 0
    safe  = exp_label.replace(',','').replace(' ','-')

    title_label_y  = table_y0 + table_frac + (0.85 * (title_frac - 0.05)) / fig_h * fig_h
    # Use fixed axis-fraction positions based on figure proportions
    label_y  = 1.0 - (0.12 / fig_h)   # 'Table N' near top
    italic_y = 1.0 - (0.45 / fig_h)   # italic title below it

    ax.text(0.0, label_y,
            f'Table {t_num}',
            transform=ax.transAxes,
            fontsize=10, fontweight='bold',
            ha='left', va='top')
    ax.text(0.0, italic_y,
            f'Classifier Performance on the {exp_label} Dataset',
            transform=ax.transAxes,
            fontsize=10, fontstyle='italic',
            ha='left', va='top')

    # APA 7 note below table
    note = (
        'Accuracy, F1 Macro, F1 Weighted, and Top-5 Accuracy are expressed as '
        'percentages. F1 Macro = unweighted mean F1 across all classes; '
        'F1 Weighted = F1 weighted by class support. '
        'CIF = Canonical Interval Forest. '
        'Dashes (—) indicate metric not available for that classifier.'
    )
    ax.text(0.0, note_frac - 0.02,
            f'Note. {note}',
            transform=ax.transAxes,
            fontsize=7.5, ha='left', va='top')

    plt.savefig(os.path.join(out_dir, f'Table_{t_num}_{safe}.png'),
                dpi=150, bbox_inches='tight')
    plt.close()
    return os.path.join(out_dir, f'Table_{t_num}_{safe}.png')


saved_tables = []
for exp in EXP_ORDER:
    df_exp = (
        df_all[df_all['Experiment']==exp]
        .set_index('Model')
        .reindex([m for m in MODEL_ORDER
                  if m in df_all[df_all['Experiment']==exp]['Model'].values])
        .reset_index()
    )
    if df_exp.empty:
        print(f'  ⬜  {exp} — no data')
        continue
    fname = apa_table_image(df_exp, exp, COMPARISON_DIR)
    saved_tables.append(fname)
    print(f'  ✅  {exp} — saved: {os.path.basename(fname)}')

(
    df_all.drop(columns=['_y_true','_y_pred','_pc_df','folder'])
    .to_csv(os.path.join(COMPARISON_DIR,'global_comparison.csv'), index=False)
)
print('\nSaved: global_comparison.csv')


  ✅  243-class — saved: Table_1_243-class.png
  ✅  1,683-class — saved: Table_2_1683-class.png
  ✅  Signer-controlled — saved: Table_3_Signer-controlled.png
  ✅  Single-signer — saved: Table_4_Single-signer.png

Saved: global_comparison.csv


In [7]:
# ── Figure 1: Grouped bar chart — Accuracy by condition and model ─────────
# APA 7 figure conventions:
#   - Figure label and title below figure
#   - No grid lines
#   - No top/right spines
#   - Legend inside figure
#   - Annotate only the highest bar per group

fig, ax = plt.subplots(figsize=(12, 5.5))

exps_avail = [e for e in EXP_ORDER
              if e in df_all['Experiment'].values]
models_avail = [m for m in MODEL_ORDER
                if m in df_all['Model'].values]

n_exp    = len(exps_avail)
n_models = len(models_avail)
x        = np.arange(n_exp)
w        = 0.18
offsets  = np.linspace(-(n_models-1)/2, (n_models-1)/2, n_models) * w

for i, model in enumerate(models_avail):
    vals = []
    for exp in exps_avail:
        row = df_all[(df_all['Experiment']==exp) &
                     (df_all['Model']==model)]
        vals.append(row['Accuracy'].values[0] if len(row) else np.nan)

    bars = ax.bar(
        x + offsets[i], vals, w,
        label     = model,
        color     = MODEL_COLORS[model],
        edgecolor = 'white',
        linewidth = 0.5,
        alpha     = 0.90,
    )

    # Annotate only the best value per model
    best_idx = np.nanargmax(vals)
    best_val = vals[best_idx]
    if not np.isnan(best_val):
        ax.text(
            x[best_idx] + offsets[i],
            best_val + 0.004,
            f'{best_val*100:.1f}%',
            ha='center', va='bottom',
            fontsize=7.5, fontweight='bold',
            color=MODEL_COLORS[model]
        )

ax.set_xticks(x)
ax.set_xticklabels(exps_avail, fontsize=10)
ax.set_ylabel('Top-1 Accuracy', fontsize=10)
ax.set_ylim(0, 0.32)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(frameon=False, fontsize=9, ncol=2, loc='upper left')
ax.set_axisbelow(False)

plt.tight_layout()
fig1_path = os.path.join(COMPARISON_DIR, 'Figure_1_accuracy_by_condition.png')
plt.savefig(fig1_path, dpi=150, bbox_inches='tight')
plt.close()
print('Saved: Figure_1_accuracy_by_condition.png')
print()
print('Figure 1. Top-1 classification accuracy for each classifier across '
      'four experimental conditions. Annotated values indicate the '
      'highest accuracy achieved by each classifier. '
      'CIF = Canonical Interval Forest.')


Saved: Figure_1_accuracy_by_condition.png

Figure 1. Top-1 classification accuracy for each classifier across four experimental conditions. Annotated values indicate the highest accuracy achieved by each classifier. CIF = Canonical Interval Forest.


In [8]:
# ── Figure 2: Top-5 Accuracy by condition ────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5.5))

for i, model in enumerate(models_avail):
    vals = []
    for exp in exps_avail:
        row = df_all[(df_all['Experiment']==exp) &
                     (df_all['Model']==model)]
        v = row['Top-5 Acc'].values[0] if len(row) else np.nan
        vals.append(v)

    bars = ax.bar(
        x + offsets[i], vals, w,
        label     = model,
        color     = MODEL_COLORS[model],
        edgecolor = 'white',
        linewidth = 0.5,
        alpha     = 0.90,
    )

    best_idx = int(np.nanargmax([v if v is not None and not
                                 (isinstance(v,float) and np.isnan(v))
                                 else -1 for v in vals]))
    best_val = vals[best_idx]
    if best_val and not np.isnan(float(best_val)):
        ax.text(
            x[best_idx] + offsets[i],
            float(best_val) + 0.006,
            f'{float(best_val)*100:.1f}%',
            ha='center', va='bottom',
            fontsize=7.5, fontweight='bold',
            color=MODEL_COLORS[model]
        )

ax.set_xticks(x)
ax.set_xticklabels(exps_avail, fontsize=10)
ax.set_ylabel('Top-5 Accuracy', fontsize=10)
ax.set_ylim(0, 0.65)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(frameon=False, fontsize=9, ncol=2, loc='upper left')

plt.tight_layout()
fig2_path = os.path.join(COMPARISON_DIR, 'Figure_2_top5_by_condition.png')
plt.savefig(fig2_path, dpi=150, bbox_inches='tight')
plt.close()
print('Saved: Figure_2_top5_by_condition.png')
print()
print('Figure 2. Top-5 classification accuracy for each classifier across '
      'four experimental conditions. Top-5 accuracy indicates the '
      'proportion of test videos where the correct sign appeared '
      'within the five highest-ranked predictions.')


Saved: Figure_2_top5_by_condition.png

Figure 2. Top-5 classification accuracy for each classifier across four experimental conditions. Top-5 accuracy indicates the proportion of test videos where the correct sign appeared within the five highest-ranked predictions.


In [9]:
# ── Figure 3: Accuracy improvement across conditions (line chart) ─────────
# Shows the trajectory of each model from full dataset to single-signer.
# Key points annotated — only start and end values to avoid crowding.

fig, ax = plt.subplots(figsize=(10, 5.5))

exp_labels_short = {
    '1,683-class'      : '1,683-class\n(full)',
    'Signer-controlled': 'Signer-\ncontrolled',
    'Single-signer'    : 'Single-\nsigner',
}
# Only the three conditions that share the same base dataset
line_exps = ['1,683-class','Signer-controlled','Single-signer']
x_pos = np.arange(len(line_exps))

for model in models_avail:
    vals = []
    for exp in line_exps:
        row = df_all[(df_all['Experiment']==exp) &
                     (df_all['Model']==model)]
        vals.append(row['Accuracy'].values[0]*100
                    if len(row) else np.nan)

    ax.plot(
        x_pos, vals,
        color     = MODEL_COLORS[model],
        marker    = MODEL_MARKERS[model],
        linewidth = 2,
        markersize= 7,
        label     = model,
    )

    # Annotate start and end only
    for idx in [0, len(line_exps)-1]:
        if not np.isnan(vals[idx]):
            ha = 'right' if idx == 0 else 'left'
            offset = -0.05 if idx == 0 else 0.05
            ax.annotate(
                f'{vals[idx]:.1f}%',
                xy     = (x_pos[idx], vals[idx]),
                xytext = (x_pos[idx]+offset, vals[idx]+0.5),
                fontsize   = 7.5,
                color      = MODEL_COLORS[model],
                fontweight = 'bold',
                ha         = ha,
            )

ax.set_xticks(x_pos)
ax.set_xticklabels([exp_labels_short.get(e,e) for e in line_exps],
                   fontsize=10)
ax.set_ylabel('Top-1 Accuracy (%)', fontsize=10)
ax.set_ylim(0, 28)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(frameon=False, fontsize=9, loc='upper left')

plt.tight_layout()
fig3_path = os.path.join(COMPARISON_DIR,
                         'Figure_3_accuracy_improvement_trajectory.png')
plt.savefig(fig3_path, dpi=150, bbox_inches='tight')
plt.close()
print('Saved: Figure_3_accuracy_improvement_trajectory.png')
print()
print('Figure 3. Classifier accuracy improvement across three experimental '
      'conditions sharing the same base vocabulary. Each condition '
      'progressively reduces intra-class signer variation. '
      'Start (1,683-class) and end (single-signer) values are annotated '
      'for each classifier.')


Saved: Figure_3_accuracy_improvement_trajectory.png

Figure 3. Classifier accuracy improvement across three experimental conditions sharing the same base vocabulary. Each condition progressively reduces intra-class signer variation. Start (1,683-class) and end (single-signer) values are annotated for each classifier.


In [10]:
# ── Figure 4: F1 Macro comparison — all conditions (grouped bar) ──────────
fig, ax = plt.subplots(figsize=(12, 5.5))

for i, model in enumerate(models_avail):
    vals = []
    for exp in exps_avail:
        row = df_all[(df_all['Experiment']==exp) &
                     (df_all['Model']==model)]
        vals.append(row['F1 Macro'].values[0] if len(row) else np.nan)

    ax.bar(
        x + offsets[i], vals, w,
        label     = model,
        color     = MODEL_COLORS[model],
        edgecolor = 'white',
        linewidth = 0.5,
        alpha     = 0.90,
    )

ax.set_xticks(x)
ax.set_xticklabels(exps_avail, fontsize=10)
ax.set_ylabel('F1 Macro', fontsize=10)
ax.set_ylim(0, 0.22)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(frameon=False, fontsize=9, ncol=2, loc='upper left')

plt.tight_layout()
fig4_path = os.path.join(COMPARISON_DIR, 'Figure_4_f1_macro_by_condition.png')
plt.savefig(fig4_path, dpi=150, bbox_inches='tight')
plt.close()
print('Saved: Figure_4_f1_macro_by_condition.png')
print()
print('Figure 4. Macro-averaged F1 score for each classifier across '
      'four experimental conditions. F1 Macro weights each class equally '
      'regardless of support, penalising classifiers that perform poorly '
      'on rare signs.')


Saved: Figure_4_f1_macro_by_condition.png

Figure 4. Macro-averaged F1 score for each classifier across four experimental conditions. F1 Macro weights each class equally regardless of support, penalising classifiers that perform poorly on rare signs.


In [11]:
# ── Table 5: Master cross-experiment summary ─────────────────────────────

pivot_acc = df_all.pivot_table(
    index='Model', columns='Experiment',
    values='Accuracy', aggfunc='first'
).reindex(index=MODEL_ORDER, columns=EXP_ORDER)

pivot_f1 = df_all.pivot_table(
    index='Model', columns='Experiment',
    values='F1 Macro', aggfunc='first'
).reindex(index=MODEL_ORDER, columns=EXP_ORDER)

rows_t5 = []
for model in MODEL_ORDER:
    row = {'Model': model}
    for exp in EXP_ORDER:
        acc = pivot_acc.loc[model, exp] if model in pivot_acc.index else np.nan
        f1  = pivot_f1.loc[model, exp]  if model in pivot_f1.index  else np.nan
        row[exp] = (
            f'{acc*100:.2f} / {f1*100:.2f}'
            if pd.notna(acc) and pd.notna(f1)
            else '— / —'
        )
    rows_t5.append(row)
df_t5 = pd.DataFrame(rows_t5)

col_labels_t5 = ['Model'] + [
    f'{e}\nAcc / F1M (%)' for e in EXP_ORDER
]

n_rows   = len(df_t5)
fig_h    = 1.4 + 0.4 + n_rows * 0.35 + 0.6
fig, ax  = plt.subplots(figsize=(15, fig_h))
ax.axis('off')

note_frac  = 0.6  / fig_h
title_frac = 1.4  / fig_h
table_frac = 1.0 - title_frac - note_frac
table_y0   = note_frac

tbl = ax.table(
    cellText  = df_t5.values,
    colLabels = col_labels_t5,
    cellLoc   = 'center',
    loc       = 'center',
    bbox      = [0, table_y0, 1, table_frac]
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)

for (row, col), cell in tbl.get_celld().items():
    cell.set_linewidth(0)
    cell.set_edgecolor('none')
    cell.set_facecolor('white')
    if row == 0:
        cell.set_text_props(fontweight='bold')

y_top    = table_y0 + table_frac
y_header = table_y0 + table_frac * (n_rows / (n_rows + 1))
y_bottom = table_y0
for y in [y_top, y_header, y_bottom]:
    ax.axhline(y=y, xmin=0.01, xmax=0.99,
               color='black', linewidth=0.8)

label_y  = 1.0 - (0.12 / fig_h)
italic_y = 1.0 - (0.45 / fig_h)

ax.text(0.0, label_y, 'Table 5',
        transform=ax.transAxes, fontsize=10,
        fontweight='bold', ha='left', va='top')
ax.text(0.0, italic_y,
        'Cross-Experiment Classifier Summary: '
        'Top-1 Accuracy and Macro F1 Across All Conditions',
        transform=ax.transAxes, fontsize=10,
        fontstyle='italic', ha='left', va='top')

ax.text(0.0, note_frac - 0.02,
        'Note. Values reported as Accuracy (%) / F1 Macro (%). '
        'Dashes indicate the experiment was not conducted for that classifier. '
        'CIF = Canonical Interval Forest. '
        'Signer-controlled = signers 11 and 13 only. '
        'Single-signer = signer 11 only.',
        transform=ax.transAxes, fontsize=7.5,
        ha='left', va='top')

plt.savefig(
    os.path.join(COMPARISON_DIR, 'Table_5_cross_experiment_summary.png'),
    dpi=150, bbox_inches='tight'
)
plt.close()
print('Saved: Table_5_cross_experiment_summary.png')


Saved: Table_5_cross_experiment_summary.png


In [12]:
# ── Export manifest ──────────────────────────────────────────────────────
print('='*65)
print('  OUTPUT FILE MANIFEST')
print('='*65)
print(f'  Directory: {COMPARISON_DIR}')
print()
all_outputs = sorted(os.listdir(COMPARISON_DIR))
tables = [f for f in all_outputs if f.startswith('Table')]
figures = [f for f in all_outputs if f.startswith('Figure')]
csvs   = [f for f in all_outputs if f.endswith('.csv')]

print('  Tables (APA 7):')
for f in tables:
    kb = os.path.getsize(os.path.join(COMPARISON_DIR,f))//1024
    print(f'    {f}  ({kb} KB)')
print()
print('  Figures:')
for f in figures:
    kb = os.path.getsize(os.path.join(COMPARISON_DIR,f))//1024
    print(f'    {f}  ({kb} KB)')
print()
print('  Data files:')
for f in csvs:
    kb = os.path.getsize(os.path.join(COMPARISON_DIR,f))//1024
    print(f'    {f}  ({kb} KB)')
print()
print('✅ Global comparison complete.')


  OUTPUT FILE MANIFEST
  Directory: /content/drive/MyDrive/Consultant/Colab_Notebooks/Obrown_Dissertation_NU_25/OBrown_DIS9300_v2/results/global_comparison_v2

  Tables (APA 7):
    Table_1_243-class.png  (79 KB)
    Table_2_1683-class.png  (75 KB)
    Table_3_Signer-controlled.png  (76 KB)
    Table_4_Single-signer.png  (80 KB)
    Table_5_154-class.png  (68 KB)
    Table_5_cross_experiment_summary.png  (84 KB)
    Table_6_master_cross_experiment.csv  (0 KB)
    Table_6_master_cross_experiment.png  (91 KB)

  Figures:
    Figure_1_accuracy_by_condition.png  (45 KB)
    Figure_2_top5_by_condition.png  (47 KB)
    Figure_3_accuracy_improvement_trajectory.png  (84 KB)
    Figure_4_f1_macro_by_condition.png  (40 KB)
    Figure_5_154class_spotlight.png  (56 KB)

  Data files:
    Table_6_master_cross_experiment.csv  (0 KB)
    all_results.csv  (2 KB)
    global_comparison.csv  (1 KB)

✅ Global comparison complete.
